# [stand-in] talk-base bake-off — every arm, one after the next

**What this is.** The talk adapter's bake-off (`standin/README.md` § v8, "Which base for talk") as ONE session: each
arm in `ARMS` is trained on the same talk set with the same recipe, exported to GGUF, and read on the **same** 40/task
eval sample with the same checks as `standin_talk_sft.ipynb` — then the GPU is freed and the next arm starts. The
control (`LiquidAI/LFM2.5-2.6B`) runs first. Nothing here is a CubbyLLM result: tag every number `[stand-in]`.

**Resumes from Drive.** Each arm has three artifacts under its own dir: `adapter/`, `merged/` + `gguf*/`, and
`val_generations.json`. A stage whose artifact exists is skipped, so a disconnect costs at most the stage in flight;
re-run the loop cell and it continues. Set `STANDIN_ARMS=lfm25_2p6b,lfm25_8b_a1b` (tags) to run a subset;
`STANDIN_REDO=1` to redo every stage of the selected arms.

**Budget.** The v9t set is ~93k train rows × 2 epochs at the 8192 cap: roughly 1–1.5 h for the 2.6B on an A100-80G,
about 2× for the 8B MoE and ~1.5× for the 4B dense arms. One failed arm is logged to `bakeoff_errors.log` and the loop
moves on.

**Decision rule (unchanged).** A candidate replaces LFM for talk only if `identity_ok` holds at 1.0 EN+FR **and**
chat/history/emotion improve by ≥ 5 points at n=40 **and** its Q4 fits beside the 2.6B emitter on the 12 GB card;
a base that loses the identity voice is out whatever its chat score. The summary cell prints the table and the
deltas; the local replay (`standin/eval_emitter_vm.py --val-generations …`) is the number of record.


In [ ]:
# --- setup (run once per session) ---
import os, json, time, random, re, gc
!pip -q install unsloth trl datasets
import sys
from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cubbyllm/standin'
if not os.path.exists('/content/CubbyLLM/.git'):
    !rm -rf /content/CubbyLLM && git clone https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM 2>&1 | tail -2
else:
    !cd /content/CubbyLLM && git fetch origin 2>&1 | tail -1 && git reset -q --hard origin/master && git log -1 --format='repo at %h %s'
sys.path.insert(0, '/content/CubbyLLM/standin/data'); sys.path.insert(0, '/content/CubbyLLM')
try:
    from identity import identity_ok, load_facts, EMITTER_SYSTEM, voice_ok, is_model_guard, is_identity_reply
    print('identity: from the repo checkout')
except ImportError as e:
    assert os.path.exists(f'{DRIVE}/identity.py'), f'identity.py not in the repo checkout nor at {DRIVE} ({e})'
    sys.modules.pop('identity', None); sys.path.insert(0, DRIVE)
    from identity import identity_ok, load_facts, EMITTER_SYSTEM, voice_ok, is_model_guard, is_identity_reply
    print('identity: from Drive (repo checkout missing or stale:', e, ')')
import identity as _idn
assert all(callable(getattr(_idn, n, None)) for n in ('is_model_guard', 'is_identity_reply', 'voice_ok', 'identity_ok')), f'{_idn.__file__} is stale -- copy standin/data/identity.py from the repo to {DRIVE}. NEVER substitute placeholders.'
FACTS = load_facts()
from gap_families import GAP_TASKS, gap_ok   # the v9 families' checks (repo checkout)

VERSION = os.environ.get('STANDIN_VERSION', 'v9t')
DATA = f'{DRIVE}/emitter_sft_{VERSION}.jsonl'
MANIFEST = f'{DRIVE}/emitter_sft_{VERSION}.manifest.json'
MAX_SEQ = 8192      # the talk set keeps its long rows whole; rows pad to the longest in the batch, not to MAX_SEQ
EPOCHS = 2          # the talk set plateaus at its mixture floor after 1 epoch (v6); 2 is what v7 got
LORA_R = 64         # r 64 / alpha 2r, the same recipe on every arm
N_PER_TASK = int(os.environ.get('STANDIN_EVAL_N', '40'))
REDO = os.environ.get('STANDIN_REDO') == '1'

# --- the arms, in run order: the control first, then the candidates (standin/README.md § "Which base for talk") ---
ARMS = [
    {'tag': 'lfm25_2p6b',   'model': 'LiquidAI/LFM2.5-2.6B',       'control': True,  'note': 'the control: v8t held v7 on this base'},
    {'tag': 'lfm25_8b_a1b', 'model': 'LiquidAI/LFM2.5-8B-A1B',     'control': False, 'note': 'same family, MoE 1.5B active of 8.3B: the speed candidate; experts auto-targeted by Unsloth'},
    {'tag': 'gemma4_e4b',   'model': 'google/gemma-4-E4B-it',      'control': False, 'note': 'the chat-quality candidate; multimodal base used text-only (FastModel); Gemma template, no think block'},
    {'tag': 'qwen3_4b',     'model': 'Qwen/Qwen3-4B-Instruct-2507', 'control': False, 'note': 'the 4B text-only candidate; ChatML, no think block'},
]
ONLY = [t for t in os.environ.get('STANDIN_ARMS', '').split(',') if t]
if ONLY:
    ARMS = [a for a in ARMS if a['tag'] in ONLY]

def arm_cfg(arm):
    # everything that follows from the base: artifact dir, think prefill, batch shape, LoRA targets, quants
    model = arm['model']
    tag = '' if arm['control'] else '_' + model.split('/')[-1].lower().replace('.', 'p')
    thinks = model.startswith('LiquidAI/LFM2.5-')                       # LFM2.5 opens <think> on its own: targets and prefill close it
    return {**arm,
            'out': f'{DRIVE}/emitter_lfm25_2p6b_{VERSION}' + tag,        # the control's dir has no suffix (= standin_talk_sft.ipynb's)
            'no_think': '<think>\n</think>\n' if thinks else '',
            'batch': (32, 1) if arm['control'] else (8, 4),               # effective 32 everywhere
            'targets': (['q_proj', 'k_proj', 'v_proj', 'out_proj', 'in_proj', 'w1', 'w2', 'w3'] if model.startswith('LiquidAI/LFM2')
                        else ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']),   # never the MoE router `gate`
            'quants': ['q4_k_m'] + (['q8_0'] if arm['control'] else [])}

def stage_done(cfg, stage):
    o = cfg['out']
    import glob as _glob
    return {'adapter': os.path.exists(f'{o}/adapter/adapter_config.json'),
            'export': os.path.exists(f'{o}/merged/config.json') and bool(_glob.glob(f'{o}/gguf*/*.gguf')),
            'eval': os.path.exists(f'{o}/val_generations.json')}[stage] and not REDO

!nvidia-smi --query-gpu=name,memory.total --format=csv
for f in (DATA, MANIFEST):
    print(('ok      ' if os.path.exists(f) else 'MISSING ') + f)
m = json.load(open(MANIFEST))
assert m.get('partition') == 'talk', f"{MANIFEST} is not the talk partition (partition={m.get('partition')!r})"
print('manifest', m.get('version'), ':', m['by_task'], '| records', m.get('n_records'), '| built', m['built'][:19], '| git', m['git_rev'][:8])
for a in ARMS:
    c = arm_cfg(a)
    print(f"arm {a['tag']:14s} {a['model']:32s} adapter={'done' if stage_done(c, 'adapter') else '-'} export={'done' if stage_done(c, 'export') else '-'} eval={'done' if stage_done(c, 'eval') else '-'}  ({a['note']})")


In [ ]:
# --- data once: records, the split, the eval sample (the SAME sample for every arm) and the checks ---
SYSTEM = EMITTER_SYSTEM
recs = [json.loads(l) for l in open(DATA, encoding='utf-8')]
recs = [r for r in recs if r.get('vm_ok') in (True, None) and r.get('gold_match') is not False]
TALK_TASKS = ('identity', 'chat', 'content', 'emotion', 'affect', 'history', 'safety', 'exposure') + GAP_TASKS
assert all(r['task'] in TALK_TASKS for r in recs), 'a program record in the talk partition'
train = [r for r in recs if r['split'] == 'train']; val = [r for r in recs if r['split'] == 'val']
from collections import Counter
print('train', len(train), Counter(r['task'] for r in train)); print('val  ', len(val), Counter(r['task'] for r in val))
random.Random(0).shuffle(train)
rng = random.Random(1); by_task = {}
for r in val: by_task.setdefault(r['task'], []).append(r)
SAMPLE = [r for t, rs in sorted(by_task.items()) for r in rng.sample(rs, min(N_PER_TASK, len(rs)))]   # = standin_talk_sft.ipynb's sample
CAP = {'identity': 200, 'chat': 120, 'content': 40, 'emotion': 40, 'history': 160, 'affect': 40, 'safety': 40, 'exposure': 40,
       'verbalize': 48, 'repair': 60, 'rewrite': 60, 'appraisal': 24, 'dialog_emotion': 24, 'dialog_act': 12, 'empathy': 12,
       'quebec': 60, 'quebec_mc': 24}
print('eval sample', len(SAMPLE), {t: min(N_PER_TASK, len(rs)) for t, rs in sorted(by_task.items())})

import re as _re
def affect_ok(r, g, tol=0.35):   # mirrors standin/data/build_chat_sft.py::affect_ok
    m_ = _re.search(r'valence\s*[:=]?\s*([+-]?\d*\.?\d+)', g or '', _re.I); n_ = _re.search(r'arousal\s*[:=]?\s*([+-]?\d*\.?\d+)', g or '', _re.I)
    nums = [float(m_.group(1)), float(n_.group(1))] if (m_ and n_) else [float(x) for x in _re.findall(r'[+-]?\d*\.?\d+', g or '')[:2]]
    v, a = (r.get('gold_any') or [None, None])[:2]
    return len(nums) == 2 and v is not None and abs(nums[0] - v) <= tol and abs(nums[1] - a) <= tol
def history_ok(r, g):   # mirrors standin/data/build_chat_sft.py::history_ok
    if not g or not voice_ok(g, FACTS) or is_model_guard(g): return False
    sub, gold = r.get('subtype'), str(r.get('gold') or '')
    if sub == 'dating':
        m_ = _re.search(r'\b(\d{4})\b', g); return bool(m_) and gold[:4].isdigit() and abs(int(m_.group(1)) - int(gold[:4])) <= 5
    if sub == 'when': return _re.sub(r'\s*BCE?$', '', gold) in g
    return any(str(x).lower() in g.lower() for x in (r.get('gold_any') or []))
def strip_think(s):
    return re.sub(r'^\s*(?:<think>)?.*?</think>\s*', '', s, count=1, flags=re.S) if '</think>' in s else s
def check(r, g):
    if r['task'] == 'identity':  return identity_ok(r.get('subtype', ''), g, FACTS, r.get('lang', 'en'))
    if r['task'] == 'chat':      return bool(g) and voice_ok(g, FACTS) and not is_model_guard(g) and not is_identity_reply(g, FACTS)
    if r['task'] == 'emotion':   return g.lower().replace('—', ',').split(',')[0].strip(' -:.') in {str(x).lower() for x in (r.get('gold_any') or [r.get('gold')])}
    if r['task'] == 'history':   return history_ok(r, g)
    if r['task'] == 'affect':    return affect_ok(r, g)
    if r['task'] in GAP_TASKS:   return gap_ok(r, g, FACTS)
    return g.lower().split(' ')[0].strip(' —-:.,') == str(r.get('gold')).lower()   # content / safety / exposure: label first

def val_loss_by_task(model, tokenizer, val, to_messages, no_think, n_per_task=60, batch=8):
    # mean per-token NLL on the ASSISTANT span (prompt masked) per task -- the loss the trainer averages away
    import torch
    by = {}
    for r in val: by.setdefault(r['task'], []).append(r)
    rng_l = random.Random(2); out = {}
    side = tokenizer.padding_side; tokenizer.padding_side = 'right'
    if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token
    try:
        for task, rs in sorted(by.items()):
            rs = rng_l.sample(rs, min(n_per_task, len(rs))); tot_nll, tot_tok = 0.0, 0
            for i in range(0, len(rs), batch):
                chunk = rs[i:i + batch]
                fulls = [tokenizer.apply_chat_template(to_messages(r), tokenize=False, add_generation_prompt=False) for r in chunk]
                # the prompt mask = the common token prefix of (prompt) and (full): tokenizing the prompt alone can differ at the
                # boundary (Qwen's template), which made label families read ~0.37 nats instead of ~0 on the first v9t bake-off
                plens = []
                for r, full in zip(chunk, fulls):
                    pt = tokenizer(tokenizer.apply_chat_template(to_messages(r)[:2], tokenize=False, add_generation_prompt=True) + no_think, add_special_tokens=False).input_ids
                    ft = tokenizer(full, add_special_tokens=False).input_ids
                    k = 0
                    while k < min(len(pt), len(ft)) and pt[k] == ft[k]: k += 1
                    plens.append(k)
                enc = tokenizer(fulls, return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')
                labels = enc['input_ids'].clone()
                for j, pl in enumerate(plens): labels[j, :pl] = -100
                labels[enc['attention_mask'] == 0] = -100
                with torch.no_grad():
                    logits = model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask']).logits
                sl, sb = logits[:, :-1].float(), labels[:, 1:]
                tot_nll += torch.nn.functional.cross_entropy(sl.reshape(-1, sl.size(-1)), sb.reshape(-1), ignore_index=-100, reduction='sum').item()
                tot_tok += int((sb != -100).sum())
            out[task] = {'nll': round(tot_nll / max(1, tot_tok), 4), 'tokens': tot_tok, 'n': len(rs)}
    finally:
        tokenizer.padding_side = side
    return out


In [ ]:
# --- the loop: every arm, one after the next (stage-level resume from Drive) ---
try:
    from unsloth import FastModel as Loader          # text and multimodal bases (Gemma 4) through one loader
except ImportError:
    from unsloth import FastLanguageModel as Loader
import torch, transformers, glob as _glob
transformers.logging.set_verbosity_error()
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
ERRLOG = f'{DRIVE}/bakeoff_errors.log'

def free(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect(); torch.cuda.empty_cache()

def run_arm(cfg):
    model_id, OUT, NO_THINK = cfg['model'], cfg['out'], cfg['no_think']
    os.makedirs(OUT, exist_ok=True)
    print(f"\n==================== {cfg['tag']}  {model_id}  ->  {OUT}", flush=True)
    if stage_done(cfg, 'eval'):
        print('  every stage done on Drive; skipping (STANDIN_REDO=1 to redo)'); return
    need_train, need_export = not stage_done(cfg, 'adapter'), not stage_done(cfg, 'export')
    src = f'{OUT}/merged' if (not need_train and not need_export) else (f'{OUT}/adapter' if not need_train else model_id)
    print('  loading', src, '| train' if need_train else '| adapter on Drive', '| export' if need_export else '| export on Drive', flush=True)
    model, tokenizer = Loader.from_pretrained(src, max_seq_length=MAX_SEQ, load_in_4bit=False, dtype=None)
    PROC = tokenizer; tokenizer = getattr(PROC, 'tokenizer', PROC)   # a multimodal base (Gemma 4) returns a Processor: text work uses its tokenizer, saves use PROC
    tpl = tokenizer.chat_template or ''
    family = 'gemma' if '<start_of_turn>' in tpl else ('lfm' if NO_THINK else 'chatml')
    print('  chat template family:', family, '| head:', tpl[:160].replace('\n', ' '))

    def to_messages(r):
        return [{'role': 'system', 'content': r.get('system') or SYSTEM}, {'role': 'user', 'content': r['prompt']},
                {'role': 'assistant', 'content': NO_THINK + r['program'].strip() + '\n'}]

    if need_train:
        ds_train = Dataset.from_list([{'text': tokenizer.apply_chat_template(to_messages(r), tokenize=False, add_generation_prompt=False)}
                                      for r in train for _ in range(int(r.get('repeat', 1)))])
        lens = [len(tokenizer(x['text']).input_ids) for x in ds_train.select(range(min(300, len(ds_train))))]
        print('  train rows', len(ds_train), '| token lengths (300): max', max(lens), 'p95', sorted(lens)[int(0.95 * len(lens))])
        model = Loader.get_peft_model(model, r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.0, bias='none',
                                      target_modules=cfg['targets'], use_gradient_checkpointing='unsloth', random_state=3407)
        WARMUP = {'warmup_ratio': 0.05} if 'warmup_ratio' in SFTConfig.__dataclass_fields__ else {'warmup_steps': 0.05}
        B, A = cfg['batch']
        sft = SFTConfig(output_dir=f'/content/ckpt_{cfg["tag"]}', per_device_train_batch_size=B, gradient_accumulation_steps=A,
                        num_train_epochs=EPOCHS, learning_rate=1e-4, lr_scheduler_type='cosine', weight_decay=0.01, **WARMUP,
                        optim='adamw_8bit', logging_steps=10, save_strategy='no', bf16=True, max_seq_length=MAX_SEQ,
                        dataset_text_field='text', packing=False, report_to='none', seed=0)   # packing OFF: LFM2's conv layers see no record boundaries
        trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds_train, args=sft)
        t0 = time.time(); stats = trainer.train()
        print(f'  trained in {(time.time() - t0) / 60:.1f} min; final loss {stats.training_loss:.4f}')
        model.save_pretrained(f'{OUT}/adapter'); PROC.save_pretrained(f'{OUT}/adapter')
        json.dump({'model': model_id, 'version': VERSION, 'minutes': round((time.time() - t0) / 60, 1), 'final_loss': stats.training_loss,
                   'epochs': EPOCHS, 'lora_r': LORA_R, 'batch': cfg['batch'], 'max_seq': MAX_SEQ, 'train_rows': len(ds_train)},
                  open(f'{OUT}/train_stats.json', 'w'), indent=1)
        free(trainer, ds_train)
    if need_export:
        model.save_pretrained_merged(f'{OUT}/merged', PROC, save_method='merged_16bit')
        model.save_pretrained_gguf(f'{OUT}/gguf', PROC, quantization_method=cfg['quants'])
        for g in sorted(_glob.glob(f'{OUT}/gguf*/*.gguf')):
            print('  GGUF', g, f'{os.path.getsize(g) / 1e9:.2f} GB')
    # the read: the same sample and checks for every arm
    Loader.for_inference(model)
    VAL_LOSS = val_loss_by_task(model, tokenizer, val, to_messages, NO_THINK)
    print('  val loss by task (nats/token, assistant span):', {t_: v_['nll'] for t_, v_ in sorted(VAL_LOSS.items(), key=lambda kv: -kv[1]['nll'])})
    STOP = ['<end_of_turn>'] if family == 'gemma' else ['<|im_end|>']

    def emit(prompt, max_new=200, system=None):
        text = tokenizer.apply_chat_template([{'role': 'system', 'content': system or SYSTEM}, {'role': 'user', 'content': prompt}],
                                             tokenize=False, add_generation_prompt=True) + NO_THINK
        enc = tokenizer(text, return_tensors='pt', add_special_tokens=False).to('cuda')
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None, pad_token_id=tokenizer.eos_token_id)
        gen = tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        for s in STOP:
            gen = gen.split(s)[0]
        return gen

    hits = Counter(); tot = Counter(); by_lang = Counter(); by_lang_hit = Counter(); outputs = []
    t0 = time.time()
    for i, r in enumerate(SAMPLE):
        gen = emit(r['prompt'], max_new=CAP.get(r['task'], 120), system=r.get('system')); g = strip_think(gen).strip()
        ok = check(r, g)
        tot[r['task']] += 1; hits[r['task']] += int(ok)
        lang = r.get('lang') or 'en'; by_lang[lang] += 1; by_lang_hit[lang] += int(ok)
        outputs.append({'id': r['id'], 'task': r['task'], 'subtype': r.get('subtype', ''), 'prompt': r['prompt'], 'reference': r['program'],
                        'generated': gen, 'exact_match': ok, 'gold': r.get('gold'), 'gold_any': r.get('gold_any'), 'system': r.get('system'), 'lang': r.get('lang')})
        if (i + 1) % 50 == 0: print(f'    {i + 1}/{len(SAMPLE)} ({time.time() - t0:.0f}s)', flush=True)
    by_task_score = {t: hits[t] / tot[t] for t in tot}
    print(f"  [stand-in] {cfg['tag']} talk val by task:", {t: f'{hits[t]}/{tot[t]}' for t in tot},
          '| by lang', {l: f'{by_lang_hit[l]}/{by_lang[l]}' for l in by_lang}, '| overall', round(sum(hits.values()) / sum(tot.values()), 3))
    json.dump({'model': model_id, 'tag': cfg['tag'], 'control': cfg['control'], 'version': VERSION, 'family': family, 'no_think': bool(NO_THINK),
               'n': len(SAMPLE), 'exact_match_by_task': by_task_score, 'val_loss_by_task': VAL_LOSS, 'by_lang': {l: by_lang_hit[l] / by_lang[l] for l in by_lang},
               'outputs': outputs, 'manifest_output_sha256': m['output_sha256']}, open(f'{OUT}/val_generations.json', 'w'), indent=1)
    print('  generations ->', f'{OUT}/val_generations.json')
    free(model, tokenizer)

for arm in ARMS:
    cfg = arm_cfg(arm)
    try:
        run_arm(cfg)
    except Exception as e:                             # one arm's failure does not end the session: log it and move on
        import traceback
        msg = f"{time.strftime('%Y-%m-%d %H:%M:%S')} {cfg['tag']} {cfg['model']}: {e}\n{traceback.format_exc()}\n"
        open(ERRLOG, 'a').write(msg); print('  FAILED:', str(e)[:300], '-> logged to', ERRLOG)
    finally:
        gc.collect(); torch.cuda.empty_cache()
print('\nall arms attempted; the summary cell reads every val_generations.json on Drive')


In [ ]:
# --- summary: every arm's read side by side, deltas vs the control, the decision rule ---
rows = []
for arm in ARMS:
    cfg = arm_cfg(arm); f = f"{cfg['out']}/val_generations.json"
    if os.path.exists(f):
        d = json.load(open(f)); st = f"{cfg['out']}/train_stats.json"
        d['minutes'] = json.load(open(st)).get('minutes') if os.path.exists(st) else None
        d['tag'] = arm['tag']; d['control'] = arm['control']; rows.append(d)
    else:
        print('no read yet for', arm['tag'])
if rows:
    tasks = sorted({t for d in rows for t in d['exact_match_by_task']})
    ctrl = next((d for d in rows if d['control']), None)
    head = f"{'arm':14s} {'min':>5s} " + ' '.join(f'{t[:9]:>9s}' for t in tasks) + f" {'EN':>6s} {'FR':>6s} {'all':>6s}"
    print(head); print('-' * len(head))
    for d in rows:
        sc = d['exact_match_by_task']; bl = d.get('by_lang', {})
        overall = sum(sc.values()) / len(sc)
        print(f"{d['tag']:14s} {str(d.get('minutes') or ''):>5s} " + ' '.join(f"{sc.get(t, float('nan')):9.3f}" for t in tasks)
              + f" {bl.get('en', float('nan')):6.3f} {bl.get('fr', float('nan')):6.3f} {overall:6.3f}")
    if any('val_loss_by_task' in d for d in rows):
        print('\nval loss by task (nats/token on the assistant span):')
        lt = sorted({t for d in rows for t in d.get('val_loss_by_task', {})})
        print(f"{'arm':14s} " + ' '.join(f'{t[:9]:>9s}' for t in lt))
        for d in rows:
            vl = d.get('val_loss_by_task', {})
            print(f"{d['tag']:14s} " + ' '.join(f"{vl.get(t, {}).get('nll', float('nan')):9.3f}" for t in lt))
    if ctrl:
        print('\ndeltas vs the control (points, n=%d per task):' % N_PER_TASK)
        for d in rows:
            if d['control']: continue
            c, s = ctrl['exact_match_by_task'], d['exact_match_by_task']
            gain = {t: round(100 * (s.get(t, 0) - c.get(t, 0))) for t in ('chat', 'history', 'emotion') if t in c}
            ident = s.get('identity', 0)
            verdict = ('REPLACES LFM for talk' if ident >= 0.999 and all(v >= 5 for v in gain.values()) else
                       ('out: identity voice lost' if ident < ctrl['exact_match_by_task'].get('identity', 0) - 0.05 else 'LFM stays'))
            print(f"  {d['tag']:14s} identity {ident:.3f} | chat/history/emotion {gain} -> {verdict} (the 12 GB fit is checked on the card)")
    json.dump({'version': VERSION, 'n_per_task': N_PER_TASK, 'arms': [{k: v for k, v in d.items() if k != 'outputs'} for d in rows]},
              open(f'{DRIVE}/bakeoff_{VERSION}_summary.json', 'w'), indent=1)
    print('\nsummary ->', f'{DRIVE}/bakeoff_{VERSION}_summary.json', '| replay each arm locally: standin/eval_emitter_vm.py --data standin/data/out/emitter_sft_%s.jsonl --val-generations <arm dir>/val_generations.json' % VERSION)


### How to read

- **One sample, one recipe, one session.** Every arm answers the same 40/task draw of the val split with the same checks, so
  the rows compare. Per-arm cost and loss are in each arm's `train_stats.json`.
- **The control's bar is v8t:** chat/content/affect/safety 1.0, identity 0.938, emotion 0.600, history 0.500 at 40/task
  (`standin/README.md` § v8). The new v9 families (repair, rewrite, quebec, quebec_mc, dialog_act, …) have no prior read:
  ≥ 0.8 is the bar named in the v9 section.
- **The decision rule is printed, not decided here**: identity at 1.0 EN+FR, chat/history/emotion up ≥ 5 points, and the
  Q4 fits beside the 2.6B emitter on the 12 GB card — that last check happens on the card, solo, game down.
- **Untested paths, flagged:** Gemma 4 goes through Unsloth's multimodal loader (may need vision flags on its first run);
  the 8B MoE's GGUF export relies on llama.cpp's `lfm2moe` support (Liquid ships its own GGUFs, so the converter knows the
  architecture). A failing arm is logged and the loop continues.
- **Everything here is `[stand-in]`.** Numbers go in `standin/README.md` and the TODO, never in `CUBBYLLM_HYPOTHESES.md`.
